# Folder 01 / file 01 — seed UCI Adult tables (also feature_refresh)

UCI Adult Census Income ([dataset](https://archive.ics.uci.edu/dataset/2/adult)): binary target `income_gt_50k` where **`>50K` = 1** and **`<=50K` = 0**. Fourteen census features; official split is `adult.data` (train) / `adult.test` (holdout).

Inlined replica of `src/n01_dev_train/n01_seed.py`. Run cells **in order** (local or Jobs). Catalog/schema/model/mode come from task env.

Seed Unity Catalog with the UCI Adult Census Income tables.

Downloads adult.data (train) and adult.test (holdout) from
https://archive.ics.uci.edu/dataset/2/adult
and writes the raw Delta table used by data checks / features.


## 1 — Imports


In [ ]:
from src.n00_shared.dataset import LABEL_COL, POSITIVE_CLASS, load_adult_pandas
from src.n00_shared.runtime import Settings, configure_mlflow, load_settings


## 2 — `_spark`


In [ ]:
def _spark():
    from pyspark.sql import SparkSession

    return SparkSession.builder.getOrCreate()


## 3 — `settings = load_settings()`


In [ ]:
settings = load_settings()


## 4 — `run()` step 1/1


In [ ]:
configure_mlflow(settings)
spark = _spark()
pdf = load_adult_pandas()
if settings.label_col != LABEL_COL:
    raise RuntimeError(f"expected label {LABEL_COL}, got {settings.label_col}")
df = spark.createDataFrame(pdf)
(
    df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(settings.raw_fq)
)
n = df.count()
pos = df.filter(f"{LABEL_COL} = 1").count()
print(
    f"seeded {settings.raw_fq} rows={n} positives(>50K)={pos} "
    f"negatives(<=50K)={n - pos} positive_class={POSITIVE_CLASS}"
)
